In [1]:
import pandas as pd

In [2]:
def load_data(csv_name: str) -> pd.DataFrame:
    """Load data from a CSV file into a DataFrame."""
    return pd.read_csv("data/" + csv_name + ".csv")

In [3]:
assessments = load_data("studentAssessment")

In [4]:
students = load_data("studentInfo")

In [5]:
withdrawn_students = students.where(students["final_result"] == "Withdrawn").dropna()

In [6]:
completing_students = students.where(students["final_result"] != "Withdrawn").dropna()

In [7]:
# students = load_data("studentInfo")
assesments = load_data("studentAssessment")
assesment_info = load_data("assessments")

In [8]:
non_final_assessments = assesment_info.where(assesment_info["assessment_type"] != "Exam").dropna()
final_assessments = assesment_info.where(assesment_info["assessment_type"] == "Exam").dropna()

In [9]:
assessments_data = assesments.merge(non_final_assessments, on="id_assessment")

In [10]:
avrage_non_final_exam = assessments_data.groupby("id_student")["score"].mean()
total_assesments_taken = assessments_data.groupby("id_student")["score"].count()

In [11]:
final_assessments_data = assesments.merge(final_assessments, on="id_assessment")
avrage_final_exam = final_assessments_data.groupby("id_student")["score"].mean()
# only 2000 students took the final exam so we will not use this data

In [12]:
students = students.merge(
    avrage_non_final_exam, left_on="id_student",
    right_index=True, how="left").merge(
        total_assesments_taken, left_on="id_student",
        right_index=True, how="left").rename(
            columns={"score_x": "avrage_assessment", "score_y": "number_of_assessments_taken"})

In [13]:
# students

In [14]:
vle = load_data("vle").drop(columns=["week_from", "week_to", "code_module", "code_presentation"])
student_vle = load_data("studentVle").merge(vle, on="id_site")

In [15]:
# student_vle.groupby(["id_student", "activity_type"])["sum_click"].mean()

In [16]:
registration_data = load_data("studentRegistration").drop_duplicates(subset=["id_student", "code_presentation"])
# registration_data 

In [17]:
students = students.merge(
    registration_data, on=["id_student", "code_presentation"], how="inner")

In [18]:
# students

In [19]:
# print(registration_data['id_student'].duplicated().sum(), students['id_student'].duplicated().sum())
# print(registration_data.groupby('id_student').size().nlargest(5))
# print(students.groupby('id_student').size().nlargest(5))


In [20]:
# Group student_vle by id_student and activity_type, calculating count, sum, and average
student_vle_agg = student_vle.groupby(['id_student', 'activity_type'])['sum_click'].agg(['count', 'sum', 'mean']).reset_index()
student_vle_agg.columns = ['id_student', 'activity_type', 'click_count', 'click_sum', 'click_average']

# Pivot to get each activity type as separate columns
student_vle_pivot = student_vle_agg.pivot(index='id_student', columns='activity_type', values=['click_count', 'click_sum', 'click_average']).fillna(0)
student_vle_pivot.columns = ['_'.join(col).strip() for col in student_vle_pivot.columns.values]

# Calculate totals across all activity types
student_vle_totals = student_vle.groupby('id_student')['sum_click'].agg(['count', 'sum', 'mean']).reset_index()
student_vle_totals.columns = ['id_student', 'total_click_count', 'total_click_sum', 'total_click_average']
student_vle_totals.set_index('id_student', inplace=True)

# Merge pivoted data with totals
student_vle_pivot = student_vle_pivot.merge(student_vle_totals, left_index=True, right_index=True)


In [21]:
# Merge onto students dataframe
students_all_vle = students.merge(student_vle_pivot, left_on='id_student', right_index=True, how='left')

In [22]:
students_total_vle = students.merge(
    student_vle_totals, left_on='id_student', right_index=True, how='left')

In [23]:
students_all_vle.to_csv("data/students_all_vle.csv", index=False)
students_total_vle.to_csv("data/students_total_vle.csv", index=False)